In [ ]:
import os, joblib, copy
import numpy as np
from sklearn.preprocessing import MinMaxScaler

def get_scaler(ruta_model, tipus="input"):
    """
    Loads a scaler (input or output) from a path and displays relevant information.

    param ruta_model (str): Path where the model is located
    param nom_model (str): Model name (for printing)
    param tipus (str): 'input' or 'output' depending on which scaler you want to load

    return: Returns the loaded scaler object
    """

    if tipus == "input":
        fitxer = os.path.join(ruta_model, "input_scaler.lib")
    elif tipus == "output":
        fitxer = os.path.join(ruta_model, "output_scaler.lib")
    else:
        raise ValueError("The parameter 'tipus' must be 'input' or 'output'")

    scaler = joblib.load(fitxer)
    return scaler


class ScalerLibBuilder:
    """
    Creates 'collage' scalers by combining columns from other MinMaxScalers.
    Always saves the .lib files in ./escalados
    """

    def __init__(self, ruta_ICE: str, ruta_PG: str):
        self.ice_in  = get_scaler(ruta_ICE, "input")
        self.ice_out = get_scaler(ruta_ICE, "output")
        self.pg_in   = get_scaler(ruta_PG,  "input")
        self.pg_out  = get_scaler(ruta_PG,  "output")

        self._av = { "ICE_input":  self.ice_in,
                     "ICE_output": self.ice_out,
                     "PG_input":   self.pg_in,
                     "PG_output":  self.pg_out }

        self.save_folder = "escalados"
        os.makedirs(self.save_folder, exist_ok=True)

    # ---------- internal utilities ----------
    def _slice_attr(self, attr_arr, idx):
        """Returns attr_arr[idx] if it exists, otherwise returns None (e.g. feature_names_in_)"""
        return attr_arr[idx] if attr_arr is not None else None

    # ---------- Main API ----------
    def create_lib(self, new_lib_name:str,
                   selections:list[tuple[str,int]]):
        """
        Builds and saves a new MinMaxScaler in the 'escalados' folder,
        copying **all** attributes from the original scaler(s)
        for each selected variable.

        Variables available in each original library (1-based indices):
        


        Args:
            new_lib_name: base name (without extension) of the resulting .lib.
            selections: list of tuples (key, position_1based), where
                - key is "ICE_input"/"ICE_output"/"PG_input"/"PG_output"
                - position_1based is the 1-based index of the variable.

        Returns:
            The complete MinMaxScaler object, with all internal attributes
            cloned and sliced according to your selection.
        """
        # Pre-allocation
        cols = len(selections)
        scale      = np.empty(cols)
        min_       = np.empty(cols)
        data_min   = np.empty(cols)
        data_max   = np.empty(cols)
        data_range = np.empty(cols)
        feat_names = []  # may remain empty

        clip_val   = None
        f_range    = None
        n_samples  = None

        # ---- slice each column ----
        for k, (key, pos1) in enumerate(selections):
            src   = self._av[key]
            i     = pos1-1                       # 0-based
            scale[k]       = src.scale_[i]
            min_[k]        = src.min_[i]
            data_min[k]    = src.data_min_[i]
            data_max[k]    = src.data_max_[i]
            data_range[k]  = src.data_range_[i]

            if hasattr(src, "feature_names_in_"):
                feat_names.append(src.feature_names_in_[i])
            else:
                feat_names = None     # at least one column without names ⇒ discard

            # copy meta only once (all columns share it)
            clip_val  = src.clip
            f_range   = src.feature_range
            n_samples = src.n_samples_seen_

        # ---- assemble the new scaler ----
        new_scaler = MinMaxScaler(feature_range=f_range, clip=clip_val, copy=True)
        # fill in manually
        new_scaler.scale_          = scale
        new_scaler.min_            = min_
        new_scaler.data_min_       = data_min
        new_scaler.data_max_       = data_max
        new_scaler.data_range_     = data_range
        new_scaler.n_features_in_  = cols
        new_scaler.n_samples_seen_ = n_samples

        if feat_names is not None:
            new_scaler.feature_names_in_ = np.array(feat_names, dtype=object)

        # ---- save ----
        out_path = os.path.join(self.save_folder, f"{new_lib_name}.lib")
        joblib.dump(new_scaler, out_path)
        print(f"✔ '{new_lib_name}.lib' saved in '{self.save_folder}/'")

        return new_scaler

# ICEv2 PGv1

        ICE_input:
          1 = Speed_rpm
          2 = m_fuel_mg
          3 = T_amb_K
          4 = p_amb_bar
          
        ICE_output:
          1 = ICE_Torque_Nm
          2 = NO_out_m_gps
          3 = NO2_out_m_gps
          4 = CO_out_m_gps
          5 = CO2_out_m_gps

        PG_input:
          1 = ICE_Speed_soll_rpm
          2 = EM2_Torque_Nm
          3 = ICE_Torque_Nm
          4 = Brake_perc

        PG_output:
          1 = Car_Speed_kmph
          2 = SOC_1

In [2]:
builder = ScalerLibBuilder(ruta_ICE="../models_markus/ICE_Model_Update_01",
                           ruta_PG="../models_markus/PG_Model_M1.1_without_EM1_Torque")



/home/usuaris.new/artur.aubach/Antic_RL_Cotxe/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.3.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [3]:

selec = [
    ("PG_output",  1), #vel_target
    ("PG_output",  1), #vel
    ("ICE_input",  2), #mf
    ("PG_input",   4), #brk
    ("ICE_input",  1), #ice_sp
]


selec2 = [
    ("ICE_input",  2), #mf
    ("ICE_input",  5), #vtg
    ("PG_input",   4), #brk
    ("ICE_input",  1), #ice_sp
]


new_scaler = builder.create_lib("input_ICEv2_PGv1", selec)
new_scaler2 = builder.create_lib("output_ICEv2_PGv1", selec)




✔ 'input_ICEv2_PGv1.lib' guardado en 'escalados/'
✔ 'output_ICEv2_PGv1.lib' guardado en 'escalados/'


# ICEv2 PGv2

        ICE_input:
          1 = Speed_rpm
          2 = m_fuel_mg
          3 = T_amb_K
          4 = p_amb_bar
          
        ICE_output:
          1 = ICE_Torque_Nm
          2 = NO_out_m_gps
          3 = NO2_out_m_gps
          4 = CO_out_m_gps
          5 = CO2_out_m_gps

        PG_input:
          1 = ICE_Speed_soll_rpm
          2 = EM2_Torque_Nm
          3 = ICE_Torque_Nm
          4 = Brake_perc

        PG_output:
          1 = Car_Speed_kmph
          2 = SOC_1

In [4]:
builder = ScalerLibBuilder(ruta_ICE="../models_markus/ICE_Model_Update_01",
                           ruta_PG="../models_markus/PG_v2")



In [5]:

selec = [
    ("PG_output",  1), #vel_target
    ("PG_output",  1), #vel
    ("ICE_input",  2), #mf
    ("PG_input",   4), #brk
    ("ICE_input",  1), #ice_sp
]


selec2 = [
    ("ICE_input",  2), #mf
    ("ICE_input",  5), #vtg
    ("PG_input",   4), #brk
    ("ICE_input",  1), #ice_sp
]


new_scaler = builder.create_lib("input_ICEv2_PGv2", selec)
new_scaler2 = builder.create_lib("output_ICEv2_PGv2", selec)




✔ 'input_ICEv2_PGv2.lib' guardado en 'escalados/'
✔ 'output_ICEv2_PGv2.lib' guardado en 'escalados/'


## RL

In [ ]:
import os, joblib, copy
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# The get_scaler function needs no changes
def get_scaler(ruta_model, tipus="input"):
    """
    Loads a scaler (input or output) from a path and displays relevant information.

    param ruta_model (str): Path where the model is located
    param nom_model (str): Model name (for printing)
    param tipus (str): 'input' or 'output' depending on which scaler you want to load

    return: Returns the loaded scaler object
    """

    if tipus == "input":
        fitxer = os.path.join(ruta_model, "input_scaler.lib")
    elif tipus == "output":
        fitxer = os.path.join(ruta_model, "output_scaler.lib")
    else:
        raise ValueError("The parameter 'tipus' must be 'input' or 'output'")

    scaler = joblib.load(fitxer)
    return scaler


class ScalerLibBuilder:
    """
    Creates 'collage' scalers by combining columns from other MinMaxScalers.
    Always saves the .lib files in ./escalados
    """

    def __init__(self, ruta_ICE: str, ruta_PG: str):
        self.ice_in  = get_scaler(ruta_ICE, "input")
        self.ice_out = get_scaler(ruta_ICE, "output")
        self.pg_in   = get_scaler(ruta_PG,  "input")
        self.pg_out  = get_scaler(ruta_PG,  "output")

In [2]:

selec = [
    ("auto", -5, 200), #vel_target
    ("auto", -5, 200), #vel
    ("auto",  3.0, 70.0), #mf
    ("auto",  0.0, 100.0), #brk
    ("auto",  0.0, 4000.0), #ice_sp
]


builder = ScalerLibBuilder(ruta_ICE="../models_markus/ICE_Model_Update_01",
                           ruta_PG="../models_markus/PG_Model_M1.1_without_EM1_Torque")




new_scaler = builder.create_lib("input_ICEv2_PGv1_RL", selec)

✔ 'input_ICEv2_PGv1_RL.lib' guardado en 'escalados/'


/home/usuaris.new/artur.aubach/Antic_RL_Cotxe/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.3.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
